# BBC News Politics Category Sub-classification
## NLP Pipeline: BGE-M3 + BERTopic (6 Modules)

**Architecture:**
- Layer 1: Data Ingestion — Raw .txt files → List[String]
- Layer 2: BGE-M3 Embeddings — 8192 token context, 1024 dims
- Layer 3: UMAP — Non-linear reduction 1024 → 5 dims
- Layer 4: HDBSCAN — Automatic density-based clustering
- Layer 5a: CountVectorizer — Removes noise words per cluster
- Layer 5b: c-TF-IDF — Distinctive keywords across clusters
- Layer 6: KeyBERTInspired — Semantic keyword refinement
- Layer 7: Output — DataFrame [filename, category, sub_category, preview]

**Final Output:** DataFrame mapping every article to its sub-category

## Imports & Environment Setup

In [ ]:
import re
import sys
import pandas as pd
import numpy as np
from loader import load_data

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

print("All imports successful")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## Data Ingestion Layer
### Loading raw Politics articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [ ]:
import os

# Path to politics articles in local directory
load_data = load_data('../data/politics')

# Load all txt files
politics_docs = []
politics_files = []

for filename in sorted(os.listdir(load_data)):
    if filename.endswith('.txt'):
        file_path = os.path.join(load_data, filename)
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
        politics_docs.append(text)
        politics_files.append(filename)

print(f"Total politics articles loaded: {len(politics_docs)}")
print(f"\nSample filename: {politics_files[0]}")
print(f"\nSample article preview:")
print(f"{politics_docs[0][:300]}")

## Data Normalization Layer
### Light cleaning only preserving full sentence structure for BGE-M3
### We only remove: encoding errors, excessive whitespace, HTML tags
### Punctuation, proper nouns, stopwords intentionally preserved

In [ ]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')

    # Remove HTML tags if present
    text = re.sub(r'<.*?>', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply light cleaning
cleaned_politics = [light_clean(doc) for doc in politics_docs]

print(f"Cleaning complete — {len(cleaned_politics)} articles processed")
print(f"\nBefore: {politics_docs[0][:200]}")
print(f"\nAfter: {cleaned_politics[0][:200]}")

##  Deduplication
### Removing duplicate articles before embedding generation
### Identical documents produce identical vectors hurts clustering quality
### Preserving filename mapping for final DataFrame output

In [ ]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_politics = []
unique_files = []

for doc, filename in zip(cleaned_politics, politics_files):
    if doc not in seen:
        seen.add(doc)
        unique_politics.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_politics)} articles")
print(f"After deduplication:  {len(unique_politics)} articles")
print(f"Duplicates removed:   {len(cleaned_politics) - len(unique_politics)}")

## Embedding Layer (BGE-M3)
### Model: BAAI/bge-m3 —> 8192 token context window, 1024 dimensional output
### Full-sequence self-attention transformer from BAAI
### Every article processed as one complete unit, no truncation
### Produces dense semantic vectors capturing meaning, context and intent

In [ ]:
# Load BGE-M3 model
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("BGE-M3 loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

##  Generate BGE-M3 Document Embeddings
### Each article is encoded as a single 1024-dimensional semantic vector
### BGE-M3 reads the entire document using full-sequence self-attention
### batch_size=32 processes 32 articles at a time to manage memory efficiently
### show_progress_bar=True lets us track encoding progress

In [ ]:
print("Generating BGE-M3 embeddings for politics articles...")
print("This will take a few minutes...")

embeddings = model.encode(
    unique_politics,
    batch_size=4, # Reduced batch size to manage GPU memory
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated successfully")
print(f"Shape: {embeddings.shape}")
print(f"Each article \u2192 {embeddings.shape[1]} dimensional vector")

## BERTopic Pipeline (All 6 Modules)
### Module 3: UMAP — compresses 1024 dims to 5, random_state=42 for stability
### Module 4: HDBSCAN — finds natural sub-categories automatically
### Module 5a: CountVectorizer — removes noise words before keyword extraction
### Module 5b: c-TF-IDF — finds distinctive keywords per cluster
### Module 6: KeyBERTInspired — refines keywords using semantic similarity

In [ ]:
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

# Module 3 — UMAP
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.01,
    random_state=42
)

# Module 4 — HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # Lowered to identify more, smaller clusters
    min_samples=3,        # Lowered to identify more, smaller clusters
    metric='euclidean',
    cluster_selection_method='eom'
)

# Module 5a — CountVectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    max_df=0.8,
    ngram_range=(1, 2)
)

# Module 5b — c-TF-IDF
ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True
)

# Module 6 — KeyBERTInspired
representation_model = KeyBERTInspired()

# Assemble full BERTopic pipeline
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    verbose=True
)

print("BERTopic pipeline assembled successfully")
print("All 6 modules configured")

## Fit BERTopic to BGE-M3 Embeddings
### Passing pre-computed BGE-M3 embeddings directly into BERTopic
### BERTopic runs all 6 modules sequentially:
### UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERTInspired
### topics_ contains sub-category assignment for every article
### probs_ contains confidence score for each assignment

In [ ]:
# Fit BERTopic using pre-computed BGE-M3 embeddings
print("Running BERTopic pipeline...")
print("UMAP \u2192 HDBSCAN \u2192 Vectorizer \u2192 c-TF-IDF \u2192 KeyBERT")
print("This may take a few minutes...")

topics, probs = topic_model.fit_transform(
    unique_politics,
    embeddings=embeddings
)

print(f"\nBERTopic complete")
print(f"Total articles processed: {len(topics)}")
print(f"Sub-categories discovered: {len(set(topics)) - 1}")
print(f"Noise articles (topic -1): {topics.count(-1)}")

### Storage: Final DataFrame
 final DataFrame as specified in the architecture. It will include the original filename, the main category ('politics'), the assigned sub-category (from BERTopic), and a preview of the article content.

In [ ]:
final_df = pd.DataFrame({
    'filename': unique_files,
    'category': 'Politics',
    'sub_category_id': topics,
    'preview': [doc[:300] + '...' for doc in unique_politics]
})

# Get human-readable topic names for sub-categories
topic_names = topic_model.get_topic_info()['Name']
# Create a mapping from topic ID to topic name
topic_id_to_name = dict(zip(topic_model.get_topic_info().Topic, topic_model.get_topic_info().Name))

# Map sub_category_id to sub_category_name
final_df['sub_category_name'] = final_df['sub_category_id'].map(topic_id_to_name)

# Handle noise articles (-1 topic_id) as 'Noise' or 'Unassigned'
final_df['sub_category_name'] = final_df['sub_category_name'].fillna('Noise')

# Reorder and select final columns
final_df = final_df[['filename', 'category', 'sub_category_id', 'sub_category_name', 'preview']]

print(f"Final DataFrame created with {len(final_df)} entries.")
display(final_df)

### Final DataFrame with Huamn labels

The final DataFrame. This will include the original filename, the main category ('Entertainment '), the assigned sub-category (human labelling ), and a preview of the article content.

In [ ]:

# POLITICS: FINAL DATAFRAME WITH NOISE RELABELLING

# Consolidation map — merge topics that talk about the same thing
politics_consolidation = {
    1:  0,   # Blair/Brown Labour internal → Labour Tory Politics
    9:  4,   # EU/Gibraltar/China → Foreign Affairs
    13: 2,   # Social legislation → Social Policy
    6:  5,   # Budget/pre-election → Election Campaign
}

politics_labels = {
    0:  "Labour Tory Politics",
    2:  "Social Policy",
    3:  "Security Terrorism",
    4:  "Foreign Affairs",
    5:  "Election Campaign",
    7:  "Scottish Devolution",
    8:  "Public Sector Reform",
    10: "Party Politics",
    11: "Electoral Reform",
    12: "Constitutional Affairs",
    14: "Political Scandals",
    15: "Law Reform",
    -1: "Unassigned"
}

# Noise remap:  70 articles
politics_noise_remap = {
    # Labour Tory Politics (0)
    "007.txt": 0,    # Fox attacks Blair's Tory lies
    "009.txt": 0,    # Campbell email row
    "075.txt": 0,    # Boris opposes mayor apology
    "154.txt": 0,    # Mayor will not retract Nazi jibe
    "186.txt": 0,    # Blair says mayor should apologise
    "196.txt": 0,    # Mandelson warning to BBC
    "228.txt": 0,    # Labour pig poster anti-Semitic
    "232.txt": 0,    # Probe into Ken Nazi jibe
    "265.txt": 0,    # Tories unveil quango blitz (dup)
    "226.txt": 0,    # Tories unveil quango blitz
    "268.txt": 0,    # More reforms ahead says Milburn
    "289.txt": 0,    # Baron Kinnock Lords debut
    "296.txt": 0,    # Tories leave door open for Archer
    "297.txt": 0,    # Mandelson warns BBC on Campbell
    "302.txt": 0,    # Galloway targets New Labour MP
    "306.txt": 0,    # Campbell returns to election team
    "308.txt": 0,    # Tory stalking horse Meyer dies
    "316.txt": 0,    # Blair moves to woo Jewish voters
    "318.txt": 0,    # Labour constituency race row
    "331.txt": 0,    # Labour attacked on Howard poster
    "332.txt": 0,    # Milburn defends poster campaign
    "400.txt": 0,    # Ex-PM Lord Callaghan dies
    "403.txt": 0,    # Voters don't trust politicians
    "405.txt": 0,    # Abortion not a poll issue Blair
    "406.txt": 0,    # Profile Gordon Brown

    # Social Policy (2)
    "056.txt": 2,    # Citizenship event for 18s
    "092.txt": 2,    # New yob targets anti-social behaviour
    "098.txt": 2,    # Ministers deny care sums wrong
    "167.txt": 2,    # Housing plans criticised by MPs
    "172.txt": 2,    # Faith schools citizenship warning
    "244.txt": 2,    # Research fears over Kelly views
    "315.txt": 2,    # CSA chief who quit still in job

    # Security Terrorism (3)
    "074.txt": 3,    # BNP leader Nick Griffin arrested
    "105.txt": 3,    # Muslim police stops more likely
    "150.txt": 3,    # Galloway plea for hostage release
    "184.txt": 3,    # Blair rejects Iraq advice calls
    "185.txt": 3,    # Police probe BNP mosque leaflet
    "188.txt": 3,    # Security papers found in street
    "324.txt": 3,    # Iraq advice claim sparks row
    "327.txt": 3,    # Goldsmith denies war advice
    "328.txt": 3,    # Goldsmith I was not leant on

    # Foreign Affairs (4)
    "037.txt": 4,    # Profile David Miliband
    "062.txt": 4,    # Muslims discuss election concerns
    "071.txt": 4,    # Visa decision every 11 minutes
    "138.txt": 4,    # EU fraud clampdown urged
    "139.txt": 4,    # Straw praises Kashmir moves
    "155.txt": 4,    # Pakistani women must not hide
    "177.txt": 4,    # Plaid MP cottage arson claim
    "389.txt": 4,    # EU rules wont stop UK spending

    # Election Campaign (5)
    "250.txt": 5,    # What the election should really be about
    "348.txt": 5,    # TV debate urged for party chiefs
    "355.txt": 5,    # Lib Dems stress Budget trust gap
    "376.txt": 5,    # MPs demand Budget leak answers
    "365.txt": 5,    # Whitehall cuts ahead of target

    # Public Sector Reform (8)
    "012.txt": 8,    # PM apology over jailings
    "076.txt": 8,    # Report attacks defence spending
    "165.txt": 8,    # Police urge pub closure power
    "178.txt": 8,    # Hatfield executives go on trial
    "198.txt": 8,    # Police chief backs drinking move
    "202.txt": 8,    # Tories opposing 24-hour drinking
    "234.txt": 8,    # Blunkett unveils policing plans
    "239.txt": 8,    # Row over police power for CSOs
    "347.txt": 8,    # Tories outlining policing plans

    # Electoral Reform (11)
    "143.txt": 11,   # Malik rejects all-black MP lists
    "238.txt": 11,   # Regiments group in poll move

    # Constitutional Affairs (12)
    "183.txt": 12,   # Former NI minister Scott dies

    # Law Reform (15)
    "038.txt": 15,   # Borders rail link campaign
    "146.txt": 15,   # Nuclear dumpsite plan attacked
    "203.txt": 15,   # New foot and mouth action urged
    "006.txt": 15,   # Errors doomed first Dome sale
}

#  Build initial DataFrame
final_df_politics = pd.DataFrame({
    'filename':         unique_files,
    'category':         'Politics',
    'sub_category_id':  topics,
    'sub_category':     [politics_labels.get(t, 'Unassigned') for t in topics],
    'confidence_score': [round(float(p), 4) if p > 0 else 0.0 for p in probs],
    'preview':          unique_politics
})

def politics_remap(row):
    sid = row['sub_category_id']
    if sid == -1:
        sid = politics_noise_remap.get(row['filename'], -1)
    sid = politics_consolidation.get(sid, sid)
    return sid, politics_labels.get(sid, 'Unassigned')

final_df_politics[['sub_category_id', 'sub_category']] = final_df_politics.apply(
    politics_remap, axis=1, result_type='expand'
)

print("\nBBC POLITICS — SUB-CATEGORY SUMMARY (fully labelled)")
print("=" * 65)

summary = final_df_politics.groupby(['sub_category_id', 'sub_category']).agg(
    article_count=('filename', 'count'),
    avg_confidence=('confidence_score', 'mean')
).reset_index()
summary['avg_confidence'] = summary['avg_confidence'].round(4)
summary = summary.sort_values('sub_category_id')

for _, row in summary.iterrows():
    print(f"\n  [{int(row['sub_category_id'])}] {row['sub_category']}")
    print(f"       Articles: {row['article_count']} | Avg Confidence: {row['avg_confidence']}")

print(f"\n{'=' * 65}")
print(f"Total articles    : {len(final_df_politics)}")
print(f"Sub-categories    : {final_df_politics['sub_category_id'].nunique()}")
print(f"Unassigned (noise): {(final_df_politics['sub_category_id'] == -1).sum()}")

display(final_df_politics[['filename', 'category', 'sub_category', 'confidence_score', 'preview']])